In [ ]:
import os
import sys
import plotly.express as px
import logging
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, Javascript
sys.path.append("../../../..")
sys.path.append("../../../../scripts")
sys.path.append("../../../../scripts/summarize/calibration")

from notebooks.notebook_styling import bellevue_theme
from input_configuration import *
from h5toDF import *
from summary_functions import *
from dictionary import *
from utils import survey_year, get_subarea, get_data

logging.disable(logging.CRITICAL)

In [ ]:
# data processing
taz_subarea = pd.read_csv(os.path.join(project_folder, districtfile))
taz_subarea['DistrictFlowName'] = taz_subarea['DistrictFlowID'].map(district_flow_name)
taz_subarea.rename(columns={'BKRCastTAZ': 'TAZ'}, inplace=True)
data_daysim = convert(os.path.join(project_folder, h5_results_file), 
                      os.path.join(project_folder, guidefile), 
                      os.path.join(project_folder, h5_results_name), stdout=False)
data_survey = convert(os.path.join(project_folder, h5_comparison_file),   # data_survey and data_fullsurvey are different in 2023, data_survey has a smaller number of records
                      os.path.join(project_folder, guidefile), 
                      os.path.join(project_folder, h5_comparison_name), stdout=False)
data_fullsurvey = convert(os.path.join(project_folder, h5_fullsurvey_file), 
                          os.path.join(project_folder, guidefile), 
                          os.path.join(project_folder, h5_fullsurvey_name), stdout=False)

data_survey['Trip_cloned']['mode'] = data_survey['Trip_cloned']['mode'].replace('TNC','Other')
data_fullsurvey['Trip']['mode'] = data_fullsurvey['Trip']['mode'].replace('TNC','Other')

# locate the ACS survey data
acs_data = os.path.join(project_folder, f'inputs/model/survey/ACS_2023.xlsx')
acs_data_bkr = os.path.join(project_folder, f'inputs/model/survey/ACS_2023_BKR.xlsx')

## Total Households and Population

In [ ]:
def tt_hh_pop(data1=data_daysim, data3=data_fullsurvey, tag='PSRC Region'):    
    name1 = 'DaysimOutputs'
    name3 = f'{survey_year}Survey' 
    th1 = data1['Household']['hhexpfac'].sum()
    th3 = data3['Household']['hhexpfac'].sum()
    tp1 = data1['Person']['psexpfac'].sum()
    tp3 = data3['Person']['psexpfac'].sum()
    ahs1 = tp1 / th1  # average household size
    ahs3 = tp3 / th3
    ph = pd.DataFrame(index = ['Total Persons', 'Total Households', 'Average Household Size'])
    ph[name1] = [tp1, th1, ahs1]
    ph[name3] = [tp3, th3, ahs3]
    ph = get_differences(ph, name1, name3, [0, 0, 3, 2, 2])

    persons_hh_acs= pd.read_excel(acs_data,sheet_name = 'Totals')
    persons_hh_acs_df = pd.DataFrame(persons_hh_acs)
    acs_persons = persons_hh_acs_df.loc[persons_hh_acs_df['DataItem']=='Persons']['Total'].sum()
    acs_hh= persons_hh_acs_df.loc[persons_hh_acs_df['DataItem']=='Households']['Total'].sum()
    acs_per_hh= persons_hh_acs_df.loc[persons_hh_acs_df['DataItem']=='PersonHH']['Total'].sum()

    ph.loc[:,'ACS'] = [acs_persons, acs_hh, acs_per_hh]
    display(ph.style.format({
            f'{name1}': '{:,.1f}',
            f'{name3}': '{:,.1f}',
            f'Difference ({name1} - {name3})': '{:,.1f}',
            f'% Difference ({name1} - {name3})': '{:,.1f}%',
            f'ACS': '{:,.1f}',
        }))

In [ ]:
tt_hh_pop(data1=data_daysim, data3=data_fullsurvey, tag='PSRC Region')

In [ ]:
import copy
_data_daysim = copy.deepcopy(data_daysim)
_data_survey = copy.deepcopy(data_survey)
_data_fullsurvey = copy.deepcopy(data_fullsurvey)
fname_tail, data_daysim_bkr, data_survey_bkr, data_fullsurvey_bkr = \
    get_data(data1=_data_daysim, data2=_data_survey, data3=_data_fullsurvey, taz_subarea=taz_subarea, if_region=False)
tt_hh_pop(data1=data_daysim_bkr, data3=data_fullsurvey_bkr, tag='BKR')

## Average Distance to Work by Person Type

In [ ]:
def avg_distance_work_pptyp(data1=data_daysim, data3=data_fullsurvey, tag='PSRC Region'):    
    #Average Distance to Work in Miles
    name1 = 'DaysimOutputs'
    name3 = f'{survey_year}Survey' 
    merge_per_hh_1 = pd.merge(data1['Person'][['hhno', 'psexpfac', 'pwpcl', 'pwtyp', 'pgend', 'pagey', 'pwaudist']],
                                data1['Household'][['hhno', 'hhtaz', 'hhparcel']],
                                on = 'hhno')

    merge_per_hh_3 = pd.merge(data3['Person'][['hhno', 'psexpfac', 'pwpcl', 'pwtyp', 'pgend', 'pagey', 'pwaudist']],
                                data3['Household'][['hhno', 'hhtaz', 'hhparcel']],
                                on = 'hhno')

    wrkrs1 = merge_per_hh_1[['pwtyp', 'psexpfac', 'hhtaz', 'pwpcl', 'hhparcel', 'pwaudist', 'pgend', 'pagey']].query('pwtyp == "Paid Full-Time Worker" or pwtyp == "Paid Part-Time Worker"')
    wrkrs3 = merge_per_hh_3[['pwtyp', 'psexpfac', 'hhtaz', 'pwpcl', 'hhparcel', 'pwaudist', 'pgend', 'pagey']].query('pwtyp == "Paid Full-Time Worker" or pwtyp == "Paid Part-Time Worker"')
    wkr_1_hzone = pd.merge(taz_subarea, wrkrs1, left_on = 'TAZ', right_on = 'hhtaz')
    wkr_3_hzone = pd.merge(taz_subarea, wrkrs3, left_on = 'TAZ', right_on = 'hhtaz')

    workers_1 = wkr_1_hzone[['pwaudist', 'hhparcel', 'pwpcl', 'pwtyp', 'pgend', 'pagey', 'psexpfac']].query('pwaudist > 0 and pwaudist < 200 and hhparcel != pwpcl').copy()
    workers_3 = wkr_3_hzone[['pwaudist', 'hhparcel', 'pwpcl', 'pwtyp', 'pgend', 'pagey', 'psexpfac']].query('pwaudist > 0 and pwaudist < 200 and hhparcel != pwpcl').copy()
    workers_1_ft = workers_1.query('pwtyp == "Paid Full-Time Worker"').copy()
    workers_3_ft = workers_3.query('pwtyp == "Paid Full-Time Worker"').copy()
    workers_1_pt = workers_1.query('pwtyp == "Paid Part-Time Worker"').copy()
    workers_3_pt = workers_3.query('pwtyp == "Paid Part-Time Worker"').copy()
    workers_1_female = workers_1.query('pgend == "Female"').copy()
    workers_3_female = workers_3.query('pgend == "Female"').copy()
    workers_1_male = workers_1.query('pgend == "Male"').copy()
    workers_3_male = workers_3.query('pgend == "Male"').copy()
    workers_1_ageund30 = workers_1.query('pagey < 30').copy()
    workers_3_ageund30 = workers_3.query('pagey < 30').copy()
    workers_1_age30to49 = workers_1.query('pagey >= 30 and pagey < 50').copy()
    workers_3_age30to49 = workers_3.query('pagey >= 30 and pagey < 50').copy()
    workers_1_age50to64 = workers_1.query('pagey >= 50 and pagey < 65').copy()
    workers_3_age50to64 = workers_3.query('pagey >= 50 and pagey < 65').copy()
    workers_1_age65up = workers_1.query('pagey >= 65').copy()
    workers_3_age65up = workers_3.query('pagey >= 65').copy()

    workers_1['Share (%)'] = workers_1['psexpfac'] / workers_1['psexpfac'].sum()
    workers_3['Share (%)'] = workers_3['psexpfac'] / workers_3['psexpfac'].sum()
    workers_1_avg_dist = weighted_average(workers_1, 'pwaudist', 'psexpfac')
    workers_3_avg_dist = weighted_average(workers_3, 'pwaudist', 'psexpfac')
    workers_1_avg_dist_ft = weighted_average(workers_1_ft, 'pwaudist', 'psexpfac')
    workers_3_avg_dist_ft = weighted_average(workers_3_ft, 'pwaudist', 'psexpfac')
    workers_1_avg_dist_pt = weighted_average(workers_1_pt, 'pwaudist', 'psexpfac')
    workers_3_avg_dist_pt = weighted_average(workers_3_pt, 'pwaudist', 'psexpfac')
    workers_1_avg_dist_f = weighted_average(workers_1_female, 'pwaudist', 'psexpfac')
    workers_3_avg_dist_f = weighted_average(workers_3_female, 'pwaudist', 'psexpfac')
    workers_1_avg_dist_m = weighted_average(workers_1_male, 'pwaudist', 'psexpfac')
    workers_3_avg_dist_m = weighted_average(workers_3_male, 'pwaudist', 'psexpfac')
    workers_1_avg_dist_ageund30 = weighted_average(workers_1_ageund30, 'pwaudist', 'psexpfac')
    workers_3_avg_dist_ageund30 = weighted_average(workers_3_ageund30, 'pwaudist', 'psexpfac')
    workers_1_avg_dist_age30to49 = weighted_average(workers_1_age30to49, 'pwaudist', 'psexpfac')
    workers_3_avg_dist_age30to49 = weighted_average(workers_3_age30to49, 'pwaudist', 'psexpfac')
    workers_1_avg_dist_age50to64 = weighted_average(workers_1_age50to64, 'pwaudist', 'psexpfac')
    workers_3_avg_dist_age50to64 = weighted_average(workers_3_age50to64, 'pwaudist', 'psexpfac')
    workers_1_avg_dist_age65up = weighted_average(workers_1_age65up, 'pwaudist', 'psexpfac')
    workers_3_avg_dist_age65up = weighted_average(workers_3_age65up, 'pwaudist', 'psexpfac')
    adw = pd.DataFrame(index = ['Total', 'Full-Time', 'Part-Time', 'Female', 'Male', 'Age Under 30', 'Age 30-49', 'Age 50-64', 'Age Over 65'])
    adw[name1]=[workers_1_avg_dist, workers_1_avg_dist_ft, workers_1_avg_dist_pt, workers_1_avg_dist_f, workers_1_avg_dist_m, workers_1_avg_dist_ageund30, workers_1_avg_dist_age30to49, workers_1_avg_dist_age50to64, workers_1_avg_dist_age65up]
    adw[name3]=[workers_3_avg_dist, workers_3_avg_dist_ft, workers_3_avg_dist_pt, workers_3_avg_dist_f, workers_3_avg_dist_m, workers_3_avg_dist_ageund30, workers_3_avg_dist_age30to49, workers_3_avg_dist_age50to64, workers_3_avg_dist_age65up]
    adw = get_differences(adw, name1, name3, 2)

    display(adw.style.format({
        name1: '{:,.1f}',
        name3: '{:,.1f}',
        f'Difference ({name1} - {name3})': '{:,.1f}',
        f'% Difference ({name1} - {name3})': '{:,.1f}%'
    }).set_caption("Average Distance to Work by Person Type"))

    wrkrslessonemi_1 = workers_1[['pwaudist', 'psexpfac']].query('pwaudist <= 1')
    wrkrslessonemi_3 = workers_3[['pwaudist', 'psexpfac']].query('pwaudist <= 1')
    workers_1_less_one_mi = 100 * wrkrslessonemi_1['psexpfac'].sum() / workers_1['psexpfac'].sum()
    workers_3_less_one_mi = 100 * wrkrslessonemi_3['psexpfac'].sum() / workers_3['psexpfac'].sum()
    wrkrsgtrtwentymi_1 = workers_1[['pwaudist', 'psexpfac']].query('pwaudist>20')
    wrkrsgtrtwentymi_3 = workers_3[['pwaudist', 'psexpfac']].query('pwaudist>20')
    workers_1_gr_twenty_mi = 100 * wrkrsgtrtwentymi_1['psexpfac'].sum() / workers_1['psexpfac'].sum()
    workers_3_gr_twenty_mi = 100 * wrkrsgtrtwentymi_3['psexpfac'].sum() / workers_3['psexpfac'].sum()

    xcl = pd.DataFrame(index = ['% Workers < 1 Mile to Work', '% Workers > 20 Miles to Work'])
    xcl[name1] = [workers_1_less_one_mi, workers_1_gr_twenty_mi]
    xcl[name3] = [workers_3_less_one_mi, workers_3_gr_twenty_mi]
    xcl = get_differences(xcl, name1, name3, 1)

    display(xcl[[name1, name3, f'Difference ({name1} - {name3})']].style.format({
        name1: '{:,.1f}%',
        name3: '{:,.1f}%',
        f'Difference ({name1} - {name3})': '{:,.1f}%'
    }).set_caption("Worker Share by Travel Distance"))

    fig = px.bar(
        adw.reset_index(),
        x='index',
        y=[name1, name3],
        barmode='group',
        labels={'index': 'Person Type', 'value': 'Avg Distance to Work (mi)', 'variable': 'Source'},
        title=f'Average Distance to Work by Person Type ({tag})'
    )
    fig.update_layout(xaxis_title='Person Type', 
                    yaxis_title='Avg Distance to Work (mi)',
                    xaxis=dict(showgrid=True), 
                    yaxis=dict(showgrid=True))
    fig.show()

In [ ]:
avg_distance_work_pptyp(data1=data_daysim, data3=data_fullsurvey, tag='PSRC Region')

In [ ]:
avg_distance_work_pptyp(data1=data_daysim_bkr, data3=data_fullsurvey_bkr, tag='BKR')

## Average Distance to School by Person Type

In [ ]:
def avg_distance_school_pptyp(data1=data_daysim, data3=data_fullsurvey, tag='PSRC Region'):    
    #Average Distance to Work in Miles
    name1 = 'DaysimOutputs'
    name3 = f'{survey_year}Survey' 

    #Average Distance to School
    students_1 = data1['Person'][['psaudist', 'psexpfac', 'pagey']].query('psaudist > 0.05 and psaudist < 200').copy()
    students_3 = data3['Person'][['psaudist', 'psexpfac', 'pagey']].query('psaudist > 0.05 and psaudist < 200').copy()
    students_1['share'] = students_1['psexpfac'] / students_1['psexpfac'].sum()
    students_3['share'] = students_3['psexpfac'] / students_3['psexpfac'].sum()
    students_1_und5 = students_1.query('pagey < 5').copy()
    students_3_und5 = students_3.query('pagey < 5').copy()
    students_1_512 = students_1.query('pagey >= 5 and pagey < 13').copy()
    students_3_512 = students_3.query('pagey >= 5 and pagey < 13').copy()
    students_1_1318 = students_1.query('pagey >= 13 and pagey < 19').copy()
    students_3_1318 = students_3.query('pagey >= 13 and pagey < 19').copy()
    students_1_19p = students_1.query('pagey >= 19').copy()
    students_3_19p = students_3.query('pagey >= 19').copy()

    students_1_avg_dist = weighted_average(students_1, 'psaudist', 'psexpfac')
    students_3_avg_dist = weighted_average(students_3, 'psaudist', 'psexpfac')
    students_1_dist_und5 = weighted_average(students_1_und5, 'psaudist', 'psexpfac')
    students_3_dist_und5 = weighted_average(students_3_und5, 'psaudist', 'psexpfac')
    students_1_dist_512 = weighted_average(students_1_512, 'psaudist', 'psexpfac')
    students_3_dist_512 = weighted_average(students_3_512, 'psaudist', 'psexpfac')
    students_1_dist_1318 = weighted_average(students_1_1318, 'psaudist', 'psexpfac')
    students_3_dist_1318 = weighted_average(students_3_1318, 'psaudist', 'psexpfac')
    students_1_dist_19p = weighted_average(students_1_19p, 'psaudist', 'psexpfac')
    students_3_dist_19p = weighted_average(students_3_19p, 'psaudist', 'psexpfac')

    ads = pd.DataFrame(index = ['All', 'Under 5', '5 to 12', '13 to 18', 'Over 19'])
    ads[name1] = [students_1_avg_dist, students_1_dist_und5, students_1_dist_512, students_1_dist_1318, students_1_dist_19p]
    ads[name3] = [students_3_avg_dist, students_3_dist_und5, students_3_dist_512, students_3_dist_1318, students_3_dist_19p]
    ads = get_differences(ads, name1, name3, 2)

    display(ads.style.format({
        name1: '{:,.1f}',
        name3: '{:,.1f}',
        f'Difference ({name1} - {name3})': '{:,.1f}',
        f'% Difference ({name1} - {name3})': '{:,.1f}%'
    }).set_caption("Average Distance to School by Person Type"))

    studlessonemi_1 = students_1[['psaudist', 'psexpfac']].query('psaudist <= 1')
    studlessonemi_3 = students_3[['psaudist', 'psexpfac']].query('psaudist <= 1')
    students_1_less_one_mi = 100 * studlessonemi_1['psexpfac'].sum() / students_1['psexpfac'].sum()
    students_3_less_one_mi = 100 * studlessonemi_3['psexpfac'].sum() / students_3['psexpfac'].sum()
    studgtrtwentymi_1 = students_1[['psaudist', 'psexpfac']].query('psaudist>20')
    studgtrtwentymi_3 = students_3[['psaudist', 'psexpfac']].query('psaudist>20')
    students_1_gr_twenty_mi = 100 * studgtrtwentymi_1['psexpfac'].sum() / students_1['psexpfac'].sum()
    students_3_gr_twenty_mi = 100 * studgtrtwentymi_3['psexpfac'].sum() / students_3['psexpfac'].sum()

    xcl = pd.DataFrame(index = ['% Students < 1 Mile to School', '% Students > 20 Miles to School'])
    xcl[name1] = [students_1_less_one_mi, students_1_gr_twenty_mi]
    xcl[name3] = [students_3_less_one_mi, students_3_gr_twenty_mi]
    xcl = get_differences(xcl, name1, name3, 1)

    display(xcl[[name1, name3, f'Difference ({name1} - {name3})']].style.format({
        name1: '{:,.1f}%',
        name3: '{:,.1f}%',
        f'Difference ({name1} - {name3})': '{:,.1f}%'
    }).set_caption("Student Share by Travel Distance"))

    fig = px.bar(
        ads.reset_index(),
        x='index',
        y=[name1, name3],
        barmode='group',
        labels={'index': 'Person Type', 'value': 'Avg Distance to School (mi)', 'variable': 'Source'},
        title=f'Average Distance to School by Person Type ({tag})'
    )
    fig.update_layout(xaxis_title='Person Type', 
                    yaxis_title='Avg Distance to School (mi)',
                    xaxis=dict(showgrid=True), 
                    yaxis=dict(showgrid=True))
    fig.show()

In [ ]:
avg_distance_school_pptyp(data1=data_daysim, data3=data_fullsurvey, tag='PSRC Region')

In [ ]:
avg_distance_school_pptyp(data1=data_daysim_bkr, data3=data_fullsurvey_bkr, tag='BKR')

## Transit Pass Ownership

In [ ]:
def transit_pass(data1=data_daysim, data3=data_fullsurvey, tag='PSRC Region'):    
    ##Transit Pass and Auto Ownership
    name1 = 'DaysimOutputs'
    name3 = f'{survey_year}Survey'    
    #Transit Pass Ownership
    #survey, data2, has ptpass values as 
    #-1: missing (they didnt ask that question on the university survey, but a lot of students have to buy a transit pass as part of their fees, so most of them probably have a pass)
    #0: no pass 
    #1-6: various types of passes, but can treat them all as 1 (yes)
    #so, set -1  and 1-6 to 1 - added by nagendra.dhakar@rsginc.com
    data3['Person'].loc[data3['Person']['ptpass'].isin([-1,1,2,3,4,5,6]), 'ptpass'] = 1
    Person_1_total = data1['Person']['psexpfac'].sum()
    Person_3_total = data3['Person']['psexpfac'].sum()

    #ttp1 = data1['Person']['ptpass'].multiply(data1['Person']['psexpfac']).sum()
    ttp1 = data1['Person'].loc[data1['Person']['ptpass'] > 0, 'psexpfac'].sum()
    ttp3 =  data3['Person'].loc[data3['Person']['ptpass'] > 0, 'psexpfac'].sum()
    ppp1 = ttp1 / Person_1_total
    ppp3 = ttp3 / Person_3_total
    tpass = pd.DataFrame(index = ['Total Transit Passes', 'Transit Passes per Person'])
    tpass[name1] = [ttp1, ppp1]
    tpass[name3] = [ttp3, ppp3]
    tpass = get_differences(tpass, name1, name3, [0, 3])

    display(tpass.style.format({
        name1: '{:,.1f}',
        name3: '{:,.1f}',
        f'Difference ({name1} - {name3})': '{:,.1f}',
        f'% Difference ({name1} - {name3})': '{:,.1f}%'
    }))

In [ ]:
transit_pass(data1=data_daysim, data3=data_fullsurvey, tag='PSRC Region')

In [ ]:
transit_pass(data1=data_daysim_bkr, data3=data_fullsurvey_bkr, tag='BKR')

## Auto Ownership

In [ ]:
def auto_ownership(data1=data_daysim, data3=data_fullsurvey, tag='PSRC Region'):    
    name1 = 'DaysimOutputs'
    name3 = f'{survey_year}Survey' 
    #Auto Ownership
    ao1 = data1['Household'][['hhvehs', 'hhexpfac']].groupby('hhvehs').sum()['hhexpfac'] / data1['Household']['hhexpfac'].sum() * 100
    # Group households of 4+ vehicles together
    for i in ao1.index.values:
        if i > 4:
            ao1[4] = ao1[4] + ao1[i]
            ao1 = ao1.drop([i])
    ao3 = data3['Household'][['hhvehs', 'hhexpfac']].groupby('hhvehs').sum()['hhexpfac'] / data3['Household']['hhexpfac'].sum() * 100
    for i in ao3.index.values:
        if i > 4: 
            ao3[4] = ao3[4] + ao3[i]
            ao3 = ao3.drop([i])
    ao = pd.DataFrame()
    ao['% of Households (' + name1 + ')'] = ao1
    ao['% of Households (' + name3 + ')'] = ao3
    ao = get_differences(ao, '% of Households (' + name1 + ')',
                             '% of Households (' + name3 + ')', 
                             1)
    aonewcol=['0', '1', '2', '3', '4+']
    ao['Number of Vehicles in Household'] = aonewcol
    ao = ao.reset_index()
    ao = ao.drop(columns = ['hhvehs'])
    ao = ao.set_index('Number of Vehicles in Household')
    
    display(ao[[f'% of Households ({name1})',
                f'% of Households ({name3})',
                f'Difference (% of Households ({name1}) - % of Households ({name3}))']].style.format({
        f'% of Households ({name1})': '{:,.1f}%',
        f'% of Households ({name3})': '{:,.1f}%',
        f'Difference (% of Households ({name1}) - % of Households ({name3}))': '{:,.1f}%'
    }))

In [ ]:
auto_ownership(data1=data_daysim, data3=data_fullsurvey, tag='PSRC Region')

In [ ]:
auto_ownership(data1=data_daysim_bkr, data3=data_fullsurvey_bkr, tag='PSRC Region')

## Share Households by Auto Ownership

In [ ]:
def hh_share_auto(data1=data_daysim, data3=data_fullsurvey, tag='PSRC Region'):    
    name1 = 'DaysimOutputs'
    name3 = f'{survey_year}Survey' 
    hh_taz1 = pd.merge(taz_subarea, data1['Household'], left_on = 'TAZ', right_on = 'hhtaz')
    hh_taz3 = pd.merge(taz_subarea, data3['Household'], left_on = 'TAZ', right_on = 'hhtaz')

    local_tag = 'County'    
    _acs_data = acs_data

    if 'BKR' in tag:
        _acs_data = acs_data_bkr
        local_tag = 'City'
        for tab in [hh_taz1, hh_taz3]:
            tab['City'] = ''            
            taz_bellevue = pd.read_csv(os.path.join(project_folder, 'inputs', 'subarea_definition', 'Bellevue_TAZ.txt'))
            taz_kirkland = pd.read_csv(os.path.join(project_folder,'inputs', 'subarea_definition', 'Kirkland_TAZ.txt'))
            taz_redmond = pd.read_csv(os.path.join(project_folder,'inputs', 'subarea_definition', 'Redmond_TAZ.txt'))
            tab.loc[tab['TAZ'].isin(taz_bellevue['TAZ']), 'City'] = 'Bellevue'
            tab.loc[tab['TAZ'].isin(taz_kirkland['TAZ']), 'City'] = 'Kirkland'
            tab.loc[tab['TAZ'].isin(taz_redmond['TAZ']), 'City'] = 'Redmond'

    aoc1 = hh_taz1[[local_tag, 'hhvehs', 'hhexpfac']].groupby([local_tag, 'hhvehs']).sum()['hhexpfac']
    aoc3 = hh_taz3[[local_tag, 'hhvehs', 'hhexpfac']].groupby([local_tag, 'hhvehs']).sum()['hhexpfac']
    autos_by_county= pd.read_excel(_acs_data, sheet_name = 'AutosCounty')
    acs_auto_share = pd.DataFrame(autos_by_county)

    #aoc2 = hh_taz2[['County', 'hhvehs', 'hhexpfac']].groupby(['County', 'hhvehs']).sum()['hhexpfac']
    counties = []
    for i in range(len(aoc1.index)):
        if aoc1.index[i][0] not in counties:
            counties.append(aoc1.index[i][0])
    aoc = pd.DataFrame(columns = ['0 Cars (' + name1 + ')', '0 Cars (' + 'ACS' + ')', '0 Cars (' + name3 + ')',
                                  '1 Car (' + name1 + ')', '1 Car (' + 'ACS' + ')', '1 Car (' + name3 + ')',
                                  '2 Cars (' + name1 + ')', '2 Cars (' + 'ACS'+ ')', '2 Cars (' + name3 + ')',
                                  '3 Cars (' + name1 + ')', '3 Cars (' + 'ACS' + ')', '3 Cars (' + name3 + ')',
                                  '4+ Cars (' + name1 + ')', '4+ Cars (' +'ACS' + ')', '4+ Cars (' + name3 + ')',], index = counties)
    aoc = aoc.infer_objects()
    aoc = aoc.fillna(float(0))
    aoc3_ = aoc3.copy(deep=True).reset_index()
    aoc3_.loc[aoc3_['hhvehs']>4, 'hhvehs'] = 4
    aoc3_ = aoc3_.groupby([local_tag, 'hhvehs']).sum()['hhexpfac']
    for aoci, name in zip([aoc1, aoc3_], [name1, name3]):
        for i in range(len(aoci.index)):
            county_group = hh_taz1[[local_tag, 'hhexpfac']].groupby(local_tag).sum()
            denominator = county_group.query(f'{local_tag} == "' + aoci.index[i][0] + '"')['hhexpfac']

            if denominator.empty:
                continue

            aoci.iloc[i] = aoci.iloc[i] * 100 / denominator
            cars = aoci.index[i][1]
            if cars == 0:
                aoc.loc[aoci.index[i][0], '0 Cars (' + name + ')'] = round(aoci.iloc[i], 2)
            elif cars == 1:
                aoc.loc[aoci.index[i][0], '1 Car (' + name + ')'] = round(aoci.iloc[i], 2)
            elif cars == 2:
                aoc.loc[aoci.index[i][0], '2 Cars (' + name + ')'] = round(aoci.iloc[i], 2)
            elif cars == 3:
                aoc.loc[aoci.index[i][0], '3 Cars (' + name + ')'] = round(aoci.iloc[i], 2)
            else:
                aoc.loc[aoci.index[i][0], '4+ Cars (' + name + ')'] += round(aoci.iloc[i], 2)

    acs0cars = (acs_auto_share['0 Cars']*100).round(2).tolist()
    acs1cars = (acs_auto_share['1 Car']*100).round(2).tolist()
    acs2cars = (acs_auto_share['2 Cars']*100).round(2).tolist()
    acs3cars = (acs_auto_share['3 Cars']*100).round(2).tolist()
    acs4cars = (acs_auto_share['4+ Cars']*100).round(2).tolist()

    aoc['0 Cars (' + 'ACS' + ')'] = acs0cars
    aoc['1 Car (' + 'ACS' + ')'] = acs1cars
    aoc['2 Cars (' + 'ACS' + ')'] = acs2cars
    aoc['3 Cars (' + 'ACS' + ')'] = acs3cars
    aoc['4+ Cars (' + 'ACS' + ')'] = acs4cars

    aoc = aoc.transpose()
    aoc.index.name = 'Number of Cars'
    aoc.columns.name = local_tag

    display(aoc.style.format('{:,.1f}%'))

In [ ]:
hh_share_auto(data1=data_daysim, data3=data_fullsurvey, tag='PSRC Region')

In [ ]:
hh_share_auto(data1=data_daysim_bkr, data3=data_fullsurvey_bkr, tag='BKR')

## Households by Income Group by Auto Ownership

In [ ]:
def create_aoi_table(hh_df, dataset_name):
    # drop records with missing income or vehicle data
    hh_df = hh_df.dropna(subset=['recinc', 'hhvehs']).copy()
    hh_df['hhvehs'] = hh_df['hhvehs'].astype(int)
    # Group by income and vehicles
    grouped = (
        hh_df
        .groupby(['recinc', 'hhvehs'])['hhexpfac']
        .sum()
        .reset_index()
    )

    # Total households by income group
    totals = (
        hh_df
        .groupby('recinc')['hhexpfac']
        .sum()
        .rename('total')
        .reset_index()
    )

    # Merge totals
    grouped = grouped.merge(totals, on='recinc')

    # Calculate percentages
    grouped['pct'] = grouped['hhexpfac'] * 100 / grouped['total']

    # Collapse 4+ vehicles
    grouped['veh_group'] = grouped['hhvehs'].apply(
        lambda x: '4+ Cars' if x >= 4 else f'{x} Car' if x == 1 else f'{x} Cars'
    )

    # Pivot table
    pivot = (
        grouped
        .pivot_table(
            index='veh_group',
            columns='recinc',
            values='pct',
            aggfunc='sum'
        )
        .round(1)
    )

    # Reorder rows
    row_order = [
        '0 Cars',
        '1 Car',
        '2 Cars',
        '3 Cars',
        '4+ Cars'
    ]

    # Rename index with dataset name
    pivot.index = [f'{idx} ({dataset_name})' for idx in pivot.index]

    return pivot

def hh_share_by_income_by_auto(data1=data_daysim, data3=data_fullsurvey, tag='PSRC Region'): 
    name1 = 'DaysimOutputs'
    name3 = f'{survey_year}Survey' 
    #Households by income group by auto ownership     
    bins = [-1, 19999, 39999, 59999, 74999, float('inf')]

    labels = [
        'Less than $20,000',
        '$20,000-$39,999',
        '$40,000-$59,999',
        '$60,000-$74,999',
        'More than $75,000'
    ]   

    data1['Household']['recinc'] = pd.cut(data1['Household']['hhincome'], bins=bins, labels=labels)
    data3['Household']['recinc'] = pd.cut(data3['Household']['hhincome'], bins=bins, labels=labels)

    aoi1 = create_aoi_table(data1['Household'], name1)
    aoi3 = create_aoi_table(data3['Household'], name3)

    # Combine tables
    aoi = pd.concat([aoi1, aoi3])

    aoi.index.name = 'Number of Cars'
    aoi.columns.name = 'Household Income'
    aoi = aoi.sort_index()    

    display(aoi.style.format('{:,.1f}%'))

In [ ]:
hh_share_by_income_by_auto(data1=data_daysim, data3=data_fullsurvey, tag='PSRC Region')

In [ ]:
hh_share_by_income_by_auto(data1=data_daysim_bkr, data3=data_fullsurvey_bkr, tag='BKR')